# Laboratorio 10 — SQL Fundamentals

Auditoría clínica, rendimiento y comparación entre DuckDB y pandas.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

candidates = [
    Path.cwd() / 'lab10',
    Path.cwd(),
    Path.cwd().parent,
]
ROOT = next(
    path for path in candidates
    if (path / 'clinical.duckdb').exists()
)
DATABASE = ROOT / 'clinical.duckdb'
connection = duckdb.connect(
    str(DATABASE), read_only=True
)
print('Directorio:', ROOT)
print('Base:', DATABASE)

Directorio: /Users/alejandrogarciacolorado/EE_CDB/lab10
Base: /Users/alejandrogarciacolorado/EE_CDB/lab10/clinical.duckdb


## Auditoría de calidad

Se verifican duplicados, fechas imposibles, valores nulos y encuentros huérfanos.

In [2]:
audit_sql = (
    ROOT / 'sql' / 'auditoria.sql'
).read_text(encoding='utf-8')
audit = connection.execute(
    audit_sql
).fetchdf()
display(audit)

,metric,result,unit
0,duplicate_patient_ids,0.0,groups
1,encounters_before_birth,0.0,rows
2,null_observations,0.0,percent
3,orphan_encounters,0.0,rows


## Medición de rendimiento

In [3]:
performance = pd.read_csv(
    ROOT / 'evidencias' / '10_performance.csv'
)
display(performance)

,version,result,repetitions,minimum_ms,median_ms,maximum_ms
0,original,143,7,3.487958,3.750792,4.307584
1,optimized,143,7,3.160125,3.320500,4.492708


## Comparación SQL contra pandas

In [4]:
comparison = pd.read_csv(
    ROOT / 'evidencias' / '11_sql_vs_pandas.csv'
)
display(comparison)

,query_id,question,sql_lines,pandas_lines,sql_median_ms,pandas_median_ms,pandas_sql_ratio,same_result
0,query_01,"Pacientes totales, vivos y fallecidos",9,6,0.523375,1.025208,1.958840,True
1,query_02,Distribución por sexo,10,10,0.998667,1.419833,1.421728,True
2,query_05,Encuentros por año,6,12,9.368167,26.267917,2.803955,True


## Conclusión

DuckDB fue adecuado para filtros, agregaciones y joins sobre millones de registros. Pandas resultó útil para trabajar con tablas resumidas.

In [5]:
connection.close()
print('Conexión cerrada correctamente.')

Conexión cerrada correctamente.
